# EasyRAG Manual Document Preparation

## Chapter position

This chapter sits before indexing. Raw files or in-memory text become canonical `Document` objects here, so mistakes at this layer tend to echo through every later stage.

## Learning question

How does manual document preparation differ from repo loading, and why do explicit IDs and logical paths matter?

## Success criteria

- create custom documents with explicit IDs and logical paths
- preview the metadata before insertion
- insert prepared `Document` objects with `ainsert_documents()`

## Source code anchors

- `easyrag.rag.orchestrator.EasyRAG.prepare_documents`
- `easyrag.rag.indexing.prepare.prepare_documents_for_insert`
- `easyrag.rag.orchestrator.EasyRAG.ainsert`
- `easyrag.rag.types.QueryParam`


## Method principles

- `easyrag.rag.orchestrator.EasyRAG.prepare_documents`: This public helper wraps the document-preparation layer. It turns raw text plus optional IDs and logical paths into canonical `Document` objects before any index write happens.
- `easyrag.rag.indexing.prepare.prepare_documents_for_insert`: This helper normalizes raw insert inputs into canonical `Document` objects. It is the lowest-level preparation path when you start from in-memory text instead of repo files.
- `easyrag.rag.orchestrator.EasyRAG.ainsert`: This is the high-level raw-text insertion entry point. It prepares documents, applies quality checks, builds storage payloads, and writes the derived artifacts into the active workspace.
- `easyrag.rag.types.QueryParam`: This dataclass is the retrieval control surface. It does not run retrieval itself; it carries mode, top-k, rewrite, MQE, rerank, and filter policy into the pipeline.

Design reason: these anchors are chosen at the raw-input to canonical-document boundary. The notebook keeps that contract explicit because later chunking, embedding, and retrieval quality are only as trustworthy as the documents and metadata produced here.


## How the code fits together

The flow in this notebook is in-memory texts -> prepared documents -> workspace insert -> grounded citations -> workspace stats. The goal is not to show every internal detail at once. The goal is to keep the boundary for this stage visible enough that later behavior still feels explainable. If a result looks odd, the intermediate objects in this notebook should tell you where to look next.

Design reason: the notebook establishes raw material first, then turns it into canonical documents, then inspects what survived that normalization boundary. That order makes it clear which later behaviors come from source quality and which come from later indexing or retrieval policy.


## Step 1: Import the public API and runtime helper

We only need the high-level `EasyRAG` API, `QueryParam`, and `run_sync` for this notebook. The rest of the behavior will come from deterministic stubs and prepared documents.


## What this cell is proving

We start by making the repo importable from the notebook runtime and loading the helpers this notebook depends on. This cell should stay quiet. It is only here to make the later examples reliable.

In [ ]:
import sys
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "notebooks" / "_utils.py").exists():
        REPO_ROOT = candidate.resolve()
        if str(REPO_ROOT) not in sys.path:
            sys.path.insert(0, str(REPO_ROOT))
        break
else:
    raise RuntimeError("Could not locate the EasyRAG repository root.")

import json
import tempfile

from easyrag import EasyRAG, QueryParam
from easyrag.support.async_utils import run_sync

This cell should only import symbols. The interesting part starts once we define the helpers and custom texts.


## Step 2: Define deterministic embedding and query-model stubs

These stubs keep the notebook zero-key runnable. The goal is not to simulate a perfect model. The goal is to make insertion and retrieval behavior stable enough for teaching.


In [ ]:
# ruff: noqa: E402,F401,F811
import sys
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "easyrag").exists():
        REPO_ROOT = candidate.resolve()
        if str(REPO_ROOT) not in sys.path:
            sys.path.insert(0, str(REPO_ROOT))
        break
else:
    raise RuntimeError("Could not locate the EasyRAG repository root.")

import json
import os
import tempfile
import time
from pathlib import Path
from pprint import pprint
from unittest import mock

from easyrag.rag import AnswerParam, EasyRAG, EvalCase, QueryParam
from easyrag.support.async_utils import run_sync
from notebooks._utils import (
    managed_demo_rag,
    print_json as _print_json,
    production_backends_ready,
    provider_ready as can_use_openai_compatible_models,
    skip_message,
    stub_embedding as _stub_embedding,
    stub_query_model as _stub_query_model,
    stub_reranker as _stub_reranker,
)

This cell defines the deterministic helpers only. There should be no runtime output yet.


## Step 3: Prepare custom `Document` objects with explicit metadata

`EasyRAG.prepare_documents()` is the easiest way to build canonical `Document` objects from in-memory text. The key idea is that you control the document IDs and logical file paths yourself. Those choices later affect ownership, upsert behavior, and citation output.

Design reason: this cell performs the real indexing-side stage transition. Calling the helper directly keeps visible where raw inputs become canonical documents, chunks, vectors, or workspace records, which is exactly the boundary you need to inspect when later retrieval looks surprising.


In [ ]:
prepared_documents = EasyRAG.prepare_documents(
    [
        "# FAQ\nEasyRAG supports local-first retrieval for documentation teams.\n",
        "# API Guide\nUse `ainsert` for raw texts and `ainsert_documents` for prepared `Document` objects.\n",
    ],
    ids=["doc::faq", "doc::api-guide"],
    file_paths=["handbook/faq.md", "handbook/api-guide.md"],
)

preview = [
    {
        "doc_id": document.metadata["doc_id"],
        "title": document.metadata["title"],
        "relative_path": document.metadata["relative_path"],
        "source_type": document.metadata["source_type"],
    }
    for document in prepared_documents
]

print(json.dumps(preview, indent=2))

You should see the exact IDs and logical file paths you supplied. This is the clearest proof that manual ingestion can preserve application-defined metadata instead of only filesystem-derived metadata.


## Step 4: Create and initialize a temporary workspace

We build the custom-documents example inside a temporary directory so the notebook remains self-contained and repeatable.

Design reason: the notebook creates an isolated `EasyRAG` workspace here so every later artifact can be explained by this one example. Making the construction explicit also surfaces which behavior comes from injected helpers, backend choice, or answer settings rather than from hidden global state.


In [ ]:
temp_dir = tempfile.TemporaryDirectory()
working_dir = temp_dir.name
workspace = "custom-documents"

rag = EasyRAG(
    working_dir=working_dir,
    workspace=workspace,
    embedding_func=_stub_embedding,
    query_model_func=_stub_query_model,
)

run_sync(rag.initialize_storages())
print(json.dumps({"working_dir": working_dir, "workspace": workspace}, indent=2))

The workspace now exists, but it does not contain any indexed knowledge yet. The next cell inserts the prepared documents you just inspected.


## Step 5: Insert the prepared documents

`ainsert_documents()` is the right entry point when you already have canonical `Document` objects and want to preserve their metadata exactly as prepared.

Design reason: this cell performs the real indexing-side stage transition. Calling the helper directly keeps visible where raw inputs become canonical documents, chunks, vectors, or workspace records, which is exactly the boundary you need to inspect when later retrieval looks surprising.


In [ ]:
prepared_insert_stats = run_sync(rag.ainsert_documents(prepared_documents))
print(json.dumps(prepared_insert_stats, indent=2))

A successful output should show nonzero document and chunk counts. At this point, the workspace contains retrievable records built from manually prepared metadata.


## Step 6: Add one direct raw-text insert for contrast

`ainsert()` is the simpler path when you do not already have `Document` objects. It still creates canonical metadata, but the normalization happens inside the orchestrator rather than before the call.

Design reason: this cell performs the real indexing-side stage transition. Calling the helper directly keeps visible where raw inputs become canonical documents, chunks, vectors, or workspace records, which is exactly the boundary you need to inspect when later retrieval looks surprising.


In [ ]:
raw_insert_stats = run_sync(
    rag.ainsert(
        "# Pricing Notes\nThe pricing workflow explains how support plans map to retrieval quotas.\n",
        ids=["doc::pricing-notes"],
        file_paths=["notes/pricing-notes.md"],
    )
)

print(json.dumps(raw_insert_stats, indent=2))

This second insert gives the workspace one more manually defined record. The important contrast is not the stats themselves. The important contrast is where the metadata normalization happened: before the call in one case, inside the call in the other.


## Step 7: Query the workspace and inspect grounded citations

Now that the custom documents are indexed, we can verify that the logical paths and titles survive all the way into retrieval output.

Design reason: this cell crosses the boundary from prepared inputs into ranked evidence. The notebook uses the structured retrieval APIs on purpose so citations, mode choice, and query metadata stay inspectable instead of disappearing behind a single search string.


In [ ]:
result = run_sync(
    rag.aquery(
        "How do I insert prepared documents?",
        QueryParam(mode="naive", rewrite_enabled=False, mqe_enabled=False),
    )
)

print(
    json.dumps(
        {
            "citations": result.citations,
            "metadata": result.metadata,
        },
        indent=2,
    )
)

You should see citation payloads that point to logical paths such as `handbook/api-guide.md`. That is the main takeaway of the notebook: manually chosen metadata is not lost during insertion or retrieval.


## Step 8: Inspect workspace stats after both insertion paths

It is useful to ask the workspace how much indexed state now exists. This gives you a compact summary of what the prepared-document path and the raw-text path created together.

Design reason: this cell inspects concrete artifacts or metadata instead of relying on a verbal description of the pipeline. Looking at the actual files, stats, citations, or contract views is what proves that the design boundary is real.


In [ ]:
stats = run_sync(rag.get_stats())
print(json.dumps(stats, indent=2))

This output should confirm that the workspace holds the combined result of both insertion styles. It is one workspace, even though the knowledge entered through two different API paths.


## Step 9: Clean up the workspace

We close the storages and remove the temporary directory so the notebook leaves no persistent state behind.


In [ ]:
run_sync(rag.finalize_storages())
temp_dir.cleanup()

print("Closed storages and removed the temporary custom-documents workspace.")

['After cleanup, the manual-ingestion walkthrough is complete. The important lesson is that EasyRAG can index knowledge from more than one entry path while still preserving stable retrieval metadata.\n', '\n', '

## Next steps

- Revisit [02_01_repo_loading_basics.ipynb](02_01_repo_loading_basics.ipynb) if you want to contrast manual preparation with repo-style loading.
- Continue with [02_04_normalization_and_cleaning.ipynb](02_04_normalization_and_cleaning.ipynb) to inspect how manual inputs are normalized before indexing.
- Read [02-data-loading-overview.md](../../docs/02-data-loading-overview.md) for the broader loading-stage map.


## Common mistakes / debugging cues

- If a later stage looks wrong, inspect `doc_id`, logical path, title, and normalized text here first.
- Do not confuse canonical `Document` preparation with actual index writes. They are different boundaries.
- Noisy or empty documents usually create problems long before retrieval. Loading and quality checks are where you catch them cheaply.

## Takeaway

The point of this notebook is to keep `in-memory texts -> prepared documents -> workspace insert -> grounded citations -> workspace stats` visible enough that the stage boundary stays readable. Once that boundary makes sense, the later notebooks stop feeling like disconnected tricks.

Continue with [02_03_pdf_and_multimodal_loading.ipynb](02_03_pdf_and_multimodal_loading.ipynb) if you want to stay in the same chapter order.